# 04 - Extra Trees Classifier

Train and evaluate an Extra Trees classifier for `performance_category` using the same split and preprocessing strategy as the Random Forest notebook for a fair comparison.

In [1]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'student_performance_processed.csv'
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA_PATH)
target = 'performance_category'
excluded = {'student_id', target, 'risk_level', 'final_marks', 'passed'}
features = [column for column in df.columns if column not in excluded]
X = df[features].copy()
y = df[target].astype(str)
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Dataset:', df.shape, '| Features:', len(features), '| Train/test:', X_train.shape, X_test.shape)

Dataset: (5000, 20) | Features: 15 | Train/test: (4000, 15) (1000, 15)


In [3]:
preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_features),
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
])
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', ExtraTreesClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, max_features='sqrt', class_weight='balanced', random_state=42, n_jobs=-1)),
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
metrics = {
    'model': 'Extra Trees',
    'accuracy': round(accuracy_score(y_test, predictions), 4),
    'precision_macro': round(precision_score(y_test, predictions, average='macro', zero_division=0), 4),
    'recall_macro': round(recall_score(y_test, predictions, average='macro', zero_division=0), 4),
    'f1_macro': round(f1_score(y_test, predictions, average='macro', zero_division=0), 4),
}
print(metrics)
print(classification_report(y_test, predictions, zero_division=0))
print('Confusion matrix:\n', confusion_matrix(y_test, predictions))

{'model': 'Extra Trees', 'accuracy': 0.516, 'precision_macro': 0.4868, 'recall_macro': 0.5848, 'f1_macro': 0.5015}
              precision    recall  f1-score   support

     average       0.64      0.50      0.56       417
   excellent       0.42      0.71      0.53       143
        good       0.58      0.43      0.49       369
        poor       0.31      0.70      0.43        71

    accuracy                           0.52      1000
   macro avg       0.49      0.58      0.50      1000
weighted avg       0.56      0.52      0.52      1000

Confusion matrix:
 [[207  30  74 106]
 [  3 102  38   0]
 [ 95 110 157   7]
 [ 20   1   0  50]]


In [4]:
model_path = MODEL_DIR / 'student_performance_extra_trees.pkl'
with model_path.open('wb') as model_file:
    pickle.dump(model, model_file)
metrics_path = MODEL_DIR / 'model_metrics.json'
saved_metrics = json.loads(metrics_path.read_text()) if metrics_path.exists() else []
saved_metrics = [row for row in saved_metrics if row.get('model') != metrics['model']]
saved_metrics.append(metrics)
metrics_path.write_text(json.dumps(saved_metrics, indent=2))
print(f'Saved model: {model_path}')
print(f'Saved evaluation metrics: {metrics_path}')

Saved model: /workspaces/LearnIQ---Personalized-Learning-Performance-Prediction-System/models/student_performance_extra_trees.pkl
Saved evaluation metrics: /workspaces/LearnIQ---Personalized-Learning-Performance-Prediction-System/models/model_metrics.json


## Conclusion

Compare the Extra Trees held-out metrics with the Random Forest result. The next notebook combines these two tree models with soft voting and selects that ensemble artifact for backend inference.